## Dataset analysis: `fact_student_academic_performance_list15.csv`

Purpose: semester-level performance facts (List15). Useful as predictors and as consistency checks vs transcript (SEMESTER_GPA).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "fact_student_academic_performance_list15.csv")
df.shape

(18528, 15)

In [2]:
df.head()

,REG_NO,ACC_NO,PROGRAM,ACADEMIC_YEAR,SEMESTER,SEMESTER_INDEX,COURSES_REGISTERED,TOTAL_CREDITS,QUALITY_POINTS,PASSED_COURSES,FAILED_COURSES,FCW_COUNT,FEX_COUNT,MEX_COUNT,SEMESTER_GPA
0,AM25B32/002,B31555,Bachelor of Science in Civil and Environmental...,2024/2025,SEM1,1,6,18,59.0,5,1,0,0,1,3.28
1,AM25B32/002,B31555,Bachelor of Science in Civil and Environmental...,2024/2025,SEM2,2,7,22,54.5,4,3,0,2,1,2.48
2,AM25B32/002,B31555,Bachelor of Science in Civil and Environmental...,2025/2026,SEM1,3,6,19,63.5,5,1,0,1,0,3.34
3,AM25B32/003,B31556,Bachelor of Science in Civil and Environmental...,2024/2025,SEM1,1,6,18,88.5,6,0,0,0,0,4.92
4,AM25B32/003,B31556,Bachelor of Science in Civil and Environmental...,2024/2025,SEM2,2,7,22,106.5,7,0,0,0,0,4.84


In [3]:
df.isna().mean().sort_values(ascending=False).head(30)

REG_NO                0.0
ACC_NO                0.0
PROGRAM               0.0
ACADEMIC_YEAR         0.0
SEMESTER              0.0
SEMESTER_INDEX        0.0
COURSES_REGISTERED    0.0
TOTAL_CREDITS         0.0
QUALITY_POINTS        0.0
PASSED_COURSES        0.0
FAILED_COURSES        0.0
FCW_COUNT             0.0
FEX_COUNT             0.0
MEX_COUNT             0.0
SEMESTER_GPA          0.0
dtype: float64

In [4]:
key = ["REG_NO", "SEMESTER_INDEX"]
df.duplicated(key).sum(), df[key].isna().any(axis=1).sum()

(np.int64(0), np.int64(0))

In [5]:
num_cols = [c for c in df.columns if c not in ["REG_NO","ACC_NO","PROGRAM","ACADEMIC_YEAR","SEMESTER"]]
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
SEMESTER_INDEX,18528.0,2.757394,1.607595,1.0,1.00,2.00,4.00,9.0
COURSES_REGISTERED,18528.0,6.481595,0.499675,6.0,6.00,6.00,7.00,7.0
TOTAL_CREDITS,18528.0,19.599957,1.653417,17.0,18.00,19.00,21.00,22.0
QUALITY_POINTS,18528.0,62.428325,15.112117,25.5,51.50,62.00,73.00,110.0
PASSED_COURSES,18528.0,4.992282,1.355117,0.0,4.00,5.00,6.00,7.0
FAILED_COURSES,18528.0,1.489313,1.301644,0.0,0.00,1.00,2.00,7.0
FCW_COUNT,18528.0,0.372139,0.594005,0.0,0.00,0.00,1.00,5.0
FEX_COUNT,18528.0,0.926921,1.122474,0.0,0.00,1.00,1.00,7.0
MEX_COUNT,18528.0,0.190253,0.429543,0.0,0.00,0.00,0.00,3.0
SEMESTER_GPA,18528.0,3.184836,0.720300,1.5,2.65,3.18,3.72,5.0


## Advanced analytics

Focus: semester performance components and how they track transcript CGPA / SEMESTER_GPA.

In [6]:
from analysis_utils import basic_profile, missingness_report, merge_to_transcript_for_cgpa

print(basic_profile(df))
missingness_report(df, top_n=20)

BasicProfile(rows=18528, cols=15, dup_rows=0, null_cells=0)


,dtype,missing_rate,missing_count,nunique
REG_NO,str,0.0,0,4992
ACC_NO,str,0.0,0,4992
PROGRAM,str,0.0,0,64
ACADEMIC_YEAR,str,0.0,0,10
SEMESTER,str,0.0,0,2
SEMESTER_INDEX,int64,0.0,0,9
COURSES_REGISTERED,int64,0.0,0,2
TOTAL_CREDITS,int64,0.0,0,6
QUALITY_POINTS,float64,0.0,0,167
PASSED_COURSES,int64,0.0,0,8


In [7]:
# Join to transcript to compare SEMESTER_GPA and components vs CGPA
trans = pd.read_csv(DATA_DIR / "student_transcript_list15.csv")
trans["CGPA"] = pd.to_numeric(trans["CGPA"], errors="coerce")

joined = merge_to_transcript_for_cgpa(df, trans, on=["REG_NO","SEMESTER_INDEX"], how="inner")

joined[[
    "CGPA",
    "SEMESTER_GPA",
    "COURSES_REGISTERED",
    "TOTAL_CREDITS",
    "PASSED_COURSES",
    "FAILED_COURSES",
    "FCW_COUNT",
    "FEX_COUNT",
    "MEX_COUNT",
]].corr(numeric_only=True)

,CGPA,SEMESTER_GPA,COURSES_REGISTERED,TOTAL_CREDITS,PASSED_COURSES,FAILED_COURSES,FCW_COUNT,FEX_COUNT,MEX_COUNT
CGPA,1.000000,0.903909,-0.000427,0.003065,0.668589,-0.696219,-0.202435,-0.666639,-0.087763
SEMESTER_GPA,0.903909,1.000000,0.001134,0.005023,0.763510,-0.794440,-0.287163,-0.709209,-0.156989
COURSES_REGISTERED,-0.000427,0.001134,1.000000,0.922329,0.289269,0.082727,0.037534,0.059673,0.042846
TOTAL_CREDITS,0.003065,0.005023,0.922329,1.000000,0.270959,0.071973,0.033155,0.052155,0.035959
PASSED_COURSES,0.668589,0.763510,0.289269,0.270959,1.000000,-0.930036,-0.395272,-0.779368,-0.235047
FAILED_COURSES,-0.696219,-0.794440,0.082727,0.071973,-0.930036,1.000000,0.425919,0.834292,0.261150
FCW_COUNT,-0.202435,-0.287163,0.037534,0.033155,-0.395272,0.425919,1.000000,-0.019519,-0.041207
FEX_COUNT,-0.666639,-0.709209,0.059673,0.052155,-0.779368,0.834292,-0.019519,1.000000,-0.058033
MEX_COUNT,-0.087763,-0.156989,0.042846,0.035959,-0.235047,0.261150,-0.041207,-0.058033,1.000000
